# Pre-entrega Módulo 4 — Clasificación de AG News con LoRA

**Alumno:** Matías Noero  
**Curso:** Data Scientist III — NLP & Deep Learning  
**Objetivo:** ajustar `distilbert-base-uncased` mediante PEFT/LoRA y compararlo con el baseline TF-IDF + LinearSVC del Módulo 3 sobre el mismo `ag_news_test.csv`.

> Este notebook mantiene el test oficial completamente aislado hasta la evaluación final. La selección del mejor checkpoint utiliza solamente una validación estratificada extraída del train oficial.

## 1. Configuración del entorno

En Google Colab, seleccionar **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU** antes de ejecutar. La primera celda instala las dependencias necesarias.

In [ ]:
%pip -q uninstall -y torchao
%pip -q install "transformers<5" "datasets<5" "peft<1" "accelerate<2" gdown

In [ ]:
import inspect
import json
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_CHECKPOINT = "distilbert-base-uncased"
CLASS_NAMES = ["World", "Sports", "Business", "Sci_Tech"]
LABEL2ID = {label: idx for idx, label in enumerate(CLASS_NAMES)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}
MAX_LENGTH = 64
VALIDATION_SIZE = 0.20

# Configuración LoRA elegida para una tarea multiclase de complejidad moderada.
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.10
TARGET_MODULES = ["q_lin", "v_lin"]

BASE_DIR = Path("/content/ds3_lora_modulo4")
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
OUTPUT_DIR = BASE_DIR / "checkpoints"
for directory in (DATA_DIR, RESULTS_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Activá una GPU en Colab antes de continuar.")
print("GPU:", torch.cuda.get_device_name(0))

### Decisiones técnicas

- **Modelo base:** `distilbert-base-uncased`, adecuado porque AG News está en inglés y reduce tiempo y memoria frente a BERT base.
- **Longitud máxima:** 64 sub-tokens. El EDA del Módulo 2 mostró un P95 de 32 tokens a nivel palabra; 64 deja margen para la tokenización WordPiece.
- **LoRA:** `r=8`, `alpha=16`, `dropout=0.10`, aplicado a las proyecciones `q_lin` y `v_lin` de la atención. Esta configuración ofrece capacidad suficiente para cuatro clases sin entrenar el Transformer completo.
- **Métrica de selección:** F1 Macro, porque permite comparar todas las clases con igual peso.

## 2. Descarga y validación de los mismos datos de los Módulos 2 y 3

Los identificadores corresponden a los archivos provistos por la cátedra. Si `gdown` no pudiera descargarlos, se pueden subir manualmente a `/content/ds3_lora_modulo4/data/` conservando los nombres indicados.

In [ ]:
import gdown

TRAIN_FILE_ID = "1TSlFi4d7Nu4K-FW1QFaR3MEe2cahfxxj"
TEST_FILE_ID = "18hrrgrXT6469bdOwSRhSqQvtKdcVPpYq"
TRAIN_PATH = DATA_DIR / "ag_news_train.csv"
TEST_PATH = DATA_DIR / "ag_news_test.csv"

if not TRAIN_PATH.exists():
    gdown.download(id=TRAIN_FILE_ID, output=str(TRAIN_PATH), quiet=False)
if not TEST_PATH.exists():
    gdown.download(id=TEST_FILE_ID, output=str(TEST_PATH), quiet=False)

print(TRAIN_PATH, TRAIN_PATH.exists())
print(TEST_PATH, TEST_PATH.exists())

In [ ]:
def load_and_validate(path, split_name):
    frame = pd.read_csv(path)
    required = {"text", "label"}
    if not required.issubset(frame.columns):
        raise ValueError(
            f"{split_name}: se esperaban {sorted(required)} y se encontraron {list(frame.columns)}"
        )
    frame = frame[["text", "label"]].copy()
    if frame.isna().any().any():
        raise ValueError(f"{split_name}: hay valores nulos")
    if (frame["text"].astype(str).str.strip() == "").any():
        raise ValueError(f"{split_name}: hay textos vacíos")
    if set(frame["label"].unique()) != set(CLASS_NAMES):
        raise ValueError(f"{split_name}: etiquetas inesperadas {sorted(frame['label'].unique())}")
    frame["labels"] = frame["label"].map(LABEL2ID).astype(int)
    return frame

train_full_df = load_and_validate(TRAIN_PATH, "train")
test_df = load_and_validate(TEST_PATH, "test")

train_df, validation_df = train_test_split(
    train_full_df,
    test_size=VALIDATION_SIZE,
    random_state=SEED,
    stratify=train_full_df["labels"],
)

print("Train:", train_df.shape)
print("Validación:", validation_df.shape)
print("Test:", test_df.shape)
display(pd.DataFrame({
    "train": train_df["label"].value_counts().sort_index(),
    "validation": validation_df["label"].value_counts().sort_index(),
    "test": test_df["label"].value_counts().sort_index(),
}))

assert len(train_full_df) == 8000, "El train no coincide con el usado en el Módulo 3"
assert len(test_df) == 2000, "El test no coincide con el usado en el Módulo 3"
assert len(set(train_full_df.index).intersection(set(test_df.index))) <= len(train_full_df)

## 3. Conversión a Hugging Face Dataset y tokenización

No se reutiliza la limpieza agresiva de TF-IDF: DistilBERT necesita conservar el orden y el contexto de la secuencia. El tokenizer aplica WordPiece, truncamiento y genera las máscaras de atención.

In [ ]:
def to_hf_dataset(frame):
    return Dataset.from_pandas(
        frame[["text", "labels"]].reset_index(drop=True),
        preserve_index=False,
    )

train_dataset = to_hf_dataset(train_df)
validation_dataset = to_hf_dataset(validation_df)
test_dataset = to_hf_dataset(test_df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_train = train_dataset.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized_validation = validation_dataset.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized_test = test_dataset.map(tokenize_batch, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
print(tokenized_train)
print("Ejemplo tokenizado:", tokenized_train[0])

## 4. Modelo Transformer y adaptadores LoRA

El modelo base queda congelado. Solo se entrenan los adaptadores de bajo rango y los componentes necesarios de la cabeza de clasificación.

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(CLASS_NAMES),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
trainable_percentage = 100 * trainable_parameters / total_parameters

parameter_summary = {
    "model_checkpoint": MODEL_CHECKPOINT,
    "total_parameters": int(total_parameters),
    "trainable_parameters": int(trainable_parameters),
    "trainable_percentage": float(trainable_percentage),
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": TARGET_MODULES,
}
print(json.dumps(parameter_summary, indent=2))
assert trainable_percentage < 3, "La proporción entrenable debería ser menor al 3%"

## 5. Métricas y configuración de entrenamiento

Se registran Accuracy, Precision, Recall y F1 tanto Macro como Weighted. F1 Macro se usa para seleccionar el mejor checkpoint.

In [ ]:
def compute_metrics(eval_prediction):
    logits, true_labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        true_labels, predictions, average="macro", zero_division=0
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        true_labels, predictions, average="weighted", zero_division=0
    )
    return {
        "accuracy": accuracy_score(true_labels, predictions),
        "precision_macro": macro_precision,
        "recall_macro": macro_recall,
        "f1_macro": macro_f1,
        "precision_weighted": weighted_precision,
        "recall_weighted": weighted_recall,
        "f1_weighted": weighted_f1,
    }

training_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.10,
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

# Compatibilidad entre versiones recientes de Transformers.
argument_names = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in argument_names:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

## 6. Fine-tuning y tiempo de entrenamiento

Esta es la celda de mayor duración. No interrumpir el entorno mientras se ejecuta.

In [ ]:
training_start = time.perf_counter()
train_output = trainer.train()
training_seconds = time.perf_counter() - training_start

print(f"Tiempo total de entrenamiento: {training_seconds:.2f} segundos")
print(f"Pérdida final de entrenamiento: {train_output.training_loss:.6f}")

In [ ]:
validation_metrics = trainer.evaluate(tokenized_validation, metric_key_prefix="validation")
print(json.dumps(validation_metrics, indent=2))

history_df = pd.DataFrame(trainer.state.log_history)
display(history_df.tail(10))

history_df.to_csv(RESULTS_DIR / "training_history.csv", index=False)

## 7. Evaluación final sobre el test oficial

El test se utiliza una sola vez, después de finalizar la selección y el entrenamiento. Sus 2.000 noticias son exactamente las mismas evaluadas por el baseline TF-IDF.

In [ ]:
test_start = time.perf_counter()
test_output = trainer.predict(tokenized_test, metric_key_prefix="test")
test_seconds = time.perf_counter() - test_start

test_predictions = np.argmax(test_output.predictions, axis=-1)
test_true = np.asarray(tokenized_test["labels"])

print(json.dumps(test_output.metrics, indent=2))
print(f"Tiempo de evaluación test: {test_seconds:.2f} segundos")

In [ ]:
report_dict = classification_report(
    test_true,
    test_predictions,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).transpose()
display(report_df.round(4))
report_df.to_csv(RESULTS_DIR / "classification_report_lora.csv")

matrix = confusion_matrix(
    test_true,
    test_predictions,
    labels=list(range(len(CLASS_NAMES))),
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.title("Matriz de confusión — DistilBERT + LoRA — AG News test")
plt.xlabel("Clase predicha")
plt.ylabel("Clase real")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix_lora.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Benchmark contra TF-IDF + LinearSVC

Los valores del baseline provienen de la Pre-entrega del Módulo 3 y fueron calculados sobre el mismo test de 2.000 noticias.

In [ ]:
baseline_metrics = {
    "Modelo": "TF-IDF + LinearSVC",
    "Precision Macro": 0.8926403988890391,
    "Recall Macro": 0.8925,
    "F1 Macro": 0.8924806441530619,
    "Accuracy": 0.8925,
    "Parámetros entrenables": np.nan,
    "Tiempo entrenamiento (s)": np.nan,
}

lora_metrics = {
    "Modelo": "DistilBERT + LoRA",
    "Precision Macro": test_output.metrics["test_precision_macro"],
    "Recall Macro": test_output.metrics["test_recall_macro"],
    "F1 Macro": test_output.metrics["test_f1_macro"],
    "Accuracy": test_output.metrics["test_accuracy"],
    "Parámetros entrenables": trainable_parameters,
    "Tiempo entrenamiento (s)": training_seconds,
}

comparison_df = pd.DataFrame([baseline_metrics, lora_metrics])
display(comparison_df.round(4))
comparison_df.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)

f1_gain = lora_metrics["F1 Macro"] - baseline_metrics["F1 Macro"]
print(f"Diferencia de F1 Macro (LoRA - baseline): {f1_gain:+.4f}")

In [ ]:
final_metrics = {
    "experiment": "AG News DistilBERT + LoRA",
    "seed": SEED,
    "train_documents": len(train_df),
    "validation_documents": len(validation_df),
    "test_documents": len(test_df),
    "max_length": MAX_LENGTH,
    "training_seconds": training_seconds,
    "test_seconds": test_seconds,
    "training_loss": float(train_output.training_loss),
    "parameters": parameter_summary,
    "validation_metrics": {key: float(value) for key, value in validation_metrics.items()},
    "test_metrics": {key: float(value) for key, value in test_output.metrics.items()},
    "baseline_macro_f1": baseline_metrics["F1 Macro"],
    "lora_macro_f1_gain": float(f1_gain),
}

with open(RESULTS_DIR / "metrics_lora.json", "w", encoding="utf-8") as file:
    json.dump(final_metrics, file, indent=2, ensure_ascii=False)

model.save_pretrained(BASE_DIR / "lora_adapter")
tokenizer.save_pretrained(BASE_DIR / "lora_adapter")

print("Resultados guardados en:", RESULTS_DIR)
print(sorted(path.name for path in RESULTS_DIR.iterdir()))

## 9. Descarga de evidencias

Esta celda comprime el reporte de clasificación, la matriz de confusión, el historial, las métricas y la tabla comparativa. Descargá el ZIP y compartilo para construir el PDF final sin inventar resultados.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/Noero_Matias_LoRA_resultados",
    "zip",
    root_dir=RESULTS_DIR,
)
print(zip_path)
files.download(zip_path)

## 10. Guía para la conclusión técnica

Completar después de ejecutar, usando los valores observados:

1. Indicar si LoRA superó el F1 Macro 0,8925 del baseline.
2. Cuantificar la diferencia absoluta de F1 y el tiempo de entrenamiento.
3. Explicar si la mejora justifica el costo de GPU.
4. Señalar las confusiones principales de ambos modelos, especialmente `Business` vs. `Sci_Tech`.
5. Mencionar como limitaciones el tamaño de la muestra, el truncamiento y el uso de un único seed.

No concluir que LoRA es mejor hasta ejecutar la evaluación final y observar las métricas reales.